# PHẦN 1: PHÂN TÍCH DỮ LIỆU KHÁM PHÁ (EDA) - HỒI QUY

## Thông tin Dataset
- **Tên dataset:** Bike Sharing Dataset (UCI Machine Learning Repository)
- **Biến mục tiêu:** `cnt` (Số lượng xe thuê mỗi giờ)
- **Kích thước mẫu:** 17,379 mẫu 
- **Số lượng đặc trưng:** 17 đặc trưng 


## 1. Thiết lập & Cấu hình

### 1.1 Import các thư viện cần thiết

In [ ]:
# Standard library imports
import sys
import os
import warnings
from pathlib import Path

# Scientific computing
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import probplot

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical models
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

### 1.2 Thiết lập Random Seed để đảm bảo tính tái hiện

In [ ]:
# Add parent directory to path for importing utils
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import set_seed, load_dataset, describe_dataset

SEED = 42
set_seed(SEED)

### 1.3 Cấu hình Visualization toàn cục

In [ ]:
# Global matplotlib configuration
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10
})

# Seaborn theme
sns.set_theme(style='whitegrid', palette='muted')
COLORS = sns.color_palette('tab10')

### 1.4 Tải Dataset

In [ ]:
# Define data paths
DATA_RAW_DIR = os.path.join('..', '..', '..', 'data', 'raw', 'regression')
DATA_FILE = os.path.join(DATA_RAW_DIR, 'hour.csv')

# Load the dataset
df = load_dataset(DATA_FILE)

# Display basic shape
n_rows, n_cols = df.shape
print(f"Shape: {n_rows:,} rows × {n_cols} columns")

## 2. Mô tả Dữ liệu

### 2.1 Tổng quan các Biến

| Biến | Vai trò | Kiểu | Mô tả |
|------|---------|------|-------|
| `instant` | ID | int | Record index (sẽ bị loại bỏ) |
| `dteday` | Ngày tháng | object | Ngày tháng (tham khảo) |
| `season` | Categorical | int | Mùa (1: Spring, 2: Summer, 3: Fall, 4: Winter) |
| `yr` | Binary | int | Năm (0: 2011, 1: 2012) |
| `mnth` | Categorical | int | Tháng (1-12) |
| `hr` | Categorical | int | Giờ (0-23) |
| `holiday` | Binary | int | Ngày lễ (0: No, 1: Yes) |
| `weekday` | Categorical | int | Ngày trong tuần (0-6) |
| `workingday` | Binary | int | Ngày làm việc (0: No, 1: Yes) |
| `weathersit` | Categorical | int | Tình trạng thời tiết (1-4) |
| `temp` | Continuous | float | Nhiệt độ (đã normalize, 0-1) |
| `atemp` | Continuous | float | Nhiệt độ cảm nhận (đã normalize) |
| `hum` | Continuous | float | Độ ẩm (đã normalize, 0-1) |
| `windspeed` | Continuous | float | Tốc độ gió (đã normalize, 0-1) |
| `casual` | **Leakage** | int | Số người thuê tạm (thành phần của cnt) |
| `registered` | **Leakage** | int | Số người thuê đăng ký (thành phần của cnt) |
| `cnt` | **Target** | int | Tổng số lượt thuê xe |

### 2.2 Phân loại Biến

In [ ]:
# Variable categorization
VARS_CONTINUOUS = ['temp', 'atemp', 'hum', 'windspeed']
VARS_BINARY = ['holiday', 'workingday', 'yr']
VARS_CATEGORICAL = ['season', 'mnth', 'hr', 'weekday', 'weathersit']
VARS_TARGET = ['cnt']
VARS_LEAKAGE = ['casual', 'registered']  # These sum up to cnt
VARS_DROP = ['instant', 'dteday']  # ID and date columns

# Summary
print("Variable Categories: ")
print(f"Continuous ({len(VARS_CONTINUOUS)}): {VARS_CONTINUOUS}")
print(f"Binary ({len(VARS_BINARY)}): {VARS_BINARY}")
print(f"Categorical ({len(VARS_CATEGORICAL)}): {VARS_CATEGORICAL}")
print(f"Target ({len(VARS_TARGET)}): {VARS_TARGET}")
print(f"Leakage ({len(VARS_LEAKAGE)}): {VARS_LEAKAGE}")
print(f"To Drop ({len(VARS_DROP)}): {VARS_DROP}")
print(f"\nTotal usable features: {len(VARS_CONTINUOUS) + len(VARS_BINARY) + len(VARS_CATEGORICAL)}")

### 2.3 Thông tin Cơ bản về Dataset

In [ ]:
# Get dataset summary from utils
summary = describe_dataset(df)

print(f"Shape: {summary['shape']}")
print(f"\nColumn Names:")
for i, col in enumerate(summary['columns'], 1):
    print(f"  {i:2d}. {col}")

# Display full info
print("\nData Types & Non-Null Counts: ")
df.info()

**Phân tích:** Dataset có 17,379 mẫu với 17 cột. Không có giá trị thiếu (tất cả 17,379 non-null). Có 4 cột float64 (continuous), 12 cột int64 (categorical/numeric), và 1 cột object (date).

### 2.4 Xem trước Dữ liệu

In [ ]:
# First 5 rows
print("First 5 Rows: ")
display(df.head())

# Last 5 rows
print("\nLast 5 Rows:")
display(df.tail())

### 2.5 Thống kê Mô tả Đầy đủ

In [ ]:
# Descriptive statistics for all columns
print("Descriptive Statistics (All Columns): ")
desc_stats = df.describe(include='all').T

# Add additional metrics
desc_stats['skewness'] = df.skew(numeric_only=True)
desc_stats['kurtosis'] = df.kurtosis(numeric_only=True)
desc_stats['missing'] = df.isnull().sum()
desc_stats['missing_pct'] = (df.isnull().sum() / len(df)) * 100

display(desc_stats.round(4))

**Phân tích:**
- **Target `cnt`**: Trung bình 189.46, median 142.0, std 181.39 → phân phối lệch phải (skewness=1.28)
- **`casual`** và **`registered`**: Có thể thấy `casual` (trung bình 35.68) + `registered` (trung bình 153.79) ≈ `cnt` (189.46) → xác nhận là data leakage
- **`temp`**: Phân bố tương đối đều (mean=0.497, std=0.193), min=0.02, max=1.0
- **Không có giá trị thiếu** (missing=0 cho tất cả cột)

### 2.6 Kiểm tra Giá trị Thiếu và Trùng lặp

In [ ]:
# Check for missing values
missing_dict = summary['missing_per_column']
total_missing = sum(missing_dict.values())

# Check for duplicates
duplicate_count = df.duplicated().sum()
duplicate_pct = (duplicate_count / len(df)) * 100

print("Data Quality Check: ")
print(f"Total Missing Values: {total_missing}")
print(f"Duplicate Rows: {duplicate_count} ({duplicate_pct:.4f}%)")

# Detailed missing by column
if total_missing > 0:
    print("\nMissing by Column:")
    for col, count in missing_dict.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(df)*100:.2f}%)")
else:
    print("→ No missing values detected")

**Kết luận:** Dataset sạch - không có missing values hay duplicate rows.

## 3. Phân tích Dữ liệu Khám phá (EDA)

### 3.1 Thống kê các Biến Liên tục

In [ ]:
# Detailed statistics for continuous features
print("Continuous Features Statistics")
cont_stats = df[VARS_CONTINUOUS].describe().T

# Add quartile ranges
cont_stats['IQR'] = cont_stats['75%'] - cont_stats['25%']
cont_stats['skewness'] = df[VARS_CONTINUOUS].skew()
cont_stats['kurtosis'] = df[VARS_CONTINUOUS].kurtosis()

# Interpret skewness
def interpret_skew(skew):
    if abs(skew) < 0.5:
        return 'Nearly symmetric'
    elif abs(skew) < 1:
        return 'Moderately skewed'
    else:
        return 'Highly skewed'

cont_stats['skew_interpretation'] = cont_stats['skewness'].apply(interpret_skew)

display(cont_stats.round(4))

**Phân tích Phân phối:**
- **`temp`**: Nearly symmetric (skew=0.004), phân bố chuẩn tốt
- **`atemp`**: Nearly symmetric (skew=-0.102), phân bố hơi lệch trái
- **`hum`**: Moderately skewed (skew=-0.086), phân bố tương đối đối xứng
- **`windspeed`**: Highly skewed (skew=1.157), lệch phải nhiều - cần xem xét transform

### 3.2 Phân tích Biến Mục tiêu (`cnt`)

In [ ]:
# Target variable analysis
target = df['cnt']

# Statistics
print("Target Variable (cnt) Statistics: ")
print(f"Count: {target.count():,}")
print(f"Mean: {target.mean():.2f}")
print(f"Median: {target.median():.2f}")
print(f"Std Dev: {target.std():.2f}")
print(f"Min: {target.min()}")
print(f"Max: {target.max()}")
print(f"Range: {target.max() - target.min()}")
print(f"\nSkewness: {target.skew():.4f}")
print(f"Kurtosis: {target.kurtosis():.4f}")

# Quartiles
q1, q2, q3 = target.quantile([0.25, 0.5, 0.75])
print(f"\nQuartiles:")
print(f"  Q1 (25%): {q1:.2f}")
print(f"  Q2 (50%): {q2:.2f}")
print(f"  Q3 (75%): {q3:.2f}")
print(f"  IQR: {q3 - q1:.2f}")

#### 3.2.1 Histogram và Boxplot của Biến Mục tiêu

In [ ]:
# Distribution plots for target
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram with KDE
sns.histplot(target, kde=True, ax=axes[0], color=COLORS[0], bins=50)
axes[0].axvline(target.mean(), color='red', linestyle='--', label=f'Mean: {target.mean():.1f}')
axes[0].axvline(target.median(), color='green', linestyle='--', label=f'Median: {target.median():.1f}')
axes[0].set_title('Distribution of Total Rentals (cnt)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Boxplot
sns.boxplot(x=target, ax=axes[1], color=COLORS[0])
axes[1].set_title('Boxplot of Total Rentals (cnt)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Count')

# Log-transformed distribution
log_target = np.log1p(target)  # log(1+x) to handle zeros
sns.histplot(log_target, kde=True, ax=axes[2], color=COLORS[1], bins=50)
axes[2].set_title('Log-transformed Distribution of cnt', fontsize=12, fontweight='bold')
axes[2].set_xlabel('log(cnt + 1)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"\nLog-transformed skewness: {log_target.skew():.4f} (Original: {target.skew():.4f})")

**Phân tích Biến Mục tiêu:**
- Phân phối lệch phải rõ rệt (skewness=1.28) với đuôi dài về phía giá trị cao
- Mean (189.5) > Median (142.0) → xác nhận lệch phải
- Có nhiều outliers ở phía trên (count > 600)
- **Khuyến nghị:** Áp dụng log-transform để giảm skewness (log skewness = 0.52 vs 1.28)

#### 3.2.2 QQ-Plot kiểm tra Tính Chuẩn

In [ ]:
# QQ-plot to check normality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original data
probplot(target, dist="norm", plot=axes[0])
axes[0].set_title('QQ-Plot: Original cnt', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Log-transformed
probplot(log_target, dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot: Log-transformed cnt', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Shapiro-Wilk test (sample for large dataset)
sample_size = min(5000, len(target))
sample_idx = np.random.choice(len(target), sample_size, replace=False)
target_sample = target.iloc[sample_idx]
log_sample = log_target.iloc[sample_idx]

stat_orig, p_orig = stats.shapiro(target_sample)
stat_log, p_log = stats.shapiro(log_sample)

print("Shapiro-Wilk Normality Test: ")
print(f"Original cnt: statistic={stat_orig:.4f}, p-value={p_orig:.2e}")
print(f"Log cnt: statistic={stat_log:.4f}, p-value={p_log:.2e}")
print("\nInterpretation: p < 0.05 → Reject normality")

**Phân tích QQ-Plot:**
- **Original cnt**: Điểm phân tán xa đường chuẩn ở cả hai đuôi → không tuân theo phân phối chuẩn
- **Log cnt**: Các điểm nằm gần đường chuẩn hơn, đặc biệt ở giữa → cải thiện đáng kể
- **Shapiro-Wilk test**: cả hai đều reject normality (p < 0.05), nhưng log-transform gần chuẩn hơn

### 3.3 Phân phối các Biến Liên tục

In [ ]:
# Distribution of continuous features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(VARS_CONTINUOUS):
    data = df[col]
    ax = axes[idx]
    
    # Histogram with KDE
    sns.histplot(data, kde=True, ax=ax, color=COLORS[idx], bins=30, alpha=0.7)
    
    # Add statistics text
    mean_val = data.mean()
    std_val = data.std()
    skew_val = data.skew()
    
    ax.axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.3f}')
    ax.set_title(f'Distribution of {col}\n(skew={skew_val:.3f})', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Distribution of Continuous Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Phân tích Phân phối Continuous:**
- **`temp`**: Phân phối gần chuẩn, trung bình ~0.5, biểu thị nhiệt độ trải đều từ lạnh đến nóng
- **`atemp`**: Tương tự temp, hơi lệch trái một chút
- **`hum` (độ ẩm)**: Phân phối đa modal với peak ở ~0.5-0.8, có một số outliers gần 0
- **`windspeed`**: Lệch phải rõ rệt, nhiều giá trị = 0 (có thể là missing được encode thành 0)

### 3.4 Phân tích các Biến Phân loại

In [ ]:
# Categorical features visualization
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Season
season_counts = df['season'].value_counts().sort_index()
season_labels = ['Spring', 'Summer', 'Fall', 'Winter']
axes[0, 0].bar(season_labels, [season_counts[i] for i in range(1, 5)], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Count by Season', fontweight='bold')
for i, v in enumerate([season_counts[i] for i in range(1, 5)]):
    axes[0, 0].text(i, v + 50, str(v), ha='center')

# Weather situation
weather_counts = df['weathersit'].value_counts().sort_index()
axes[0, 1].bar(weather_counts.index, weather_counts.values, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Count by Weather Situation', fontweight='bold')
axes[0, 1].set_xlabel('Weather (1=Clear, 2=Mist, 3=Light Rain/Snow, 4=Heavy Rain/Snow)')

# Hour
hour_counts = df['hr'].value_counts().sort_index()
axes[1, 0].bar(hour_counts.index, hour_counts.values, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Count by Hour of Day', fontweight='bold')
axes[1, 0].set_xlabel('Hour')

# Month
month_counts = df['mnth'].value_counts().sort_index()
axes[1, 1].bar(month_counts.index, month_counts.values, color='gold', edgecolor='black')
axes[1, 1].set_title('Count by Month', fontweight='bold')
axes[1, 1].set_xlabel('Month')

# Working day vs Holiday
work_counts = df['workingday'].value_counts()
axes[2, 0].pie(work_counts.values, labels=['Non-working', 'Working'], autopct='%1.1f%%', startangle=90)
axes[2, 0].set_title('Working Day Distribution', fontweight='bold')

# Year
year_counts = df['yr'].value_counts().sort_index()
year_labels = ['2011', '2012']
axes[2, 1].bar(year_labels, [year_counts[0], year_counts[1]], color='mediumpurple', edgecolor='black')
axes[2, 1].set_title('Count by Year', fontweight='bold')
for i, v in enumerate([year_counts[0], year_counts[1]]):
    axes[2, 1].text(i, v + 100, str(v), ha='center')

plt.tight_layout()
plt.show()

# Print statistics
print("Categorical Features Statistics")
print(f"\nSeason distribution:")
for i, label in enumerate(season_labels, 1):
    print(f"  {label}: {season_counts[i]} samples ({season_counts[i]/len(df)*100:.1f}%)")

print(f"\nYear distribution:")
print(f"  2011: {year_counts[0]} samples ({year_counts[0]/len(df)*100:.1f}%)")
print(f"  2012: {year_counts[1]} samples ({year_counts[1]/len(df)*100:.1f}%)")

print(f"\nWorking day: {work_counts[1]} working days ({work_counts[1]/len(df)*100:.1f}%)")

**Phân tích Biến Phân loại:**
- **Season**: Phân bố đều 4 mùa (~4,300 samples/mùa), mùa Spring ít hơn một chút (4,268)
- **Weather**: Hầu hết là Clear/Partly cloudy (65.9%), rất ít Heavy Rain (3 samples)
- **Hour**: Phân bố đều (mỗi giờ ~724 samples) vì dữ liệu theo giờ
- **Year**: 2011 (8,709 samples, 50.1%) vs 2012 (8,670 samples, 49.9%) - cân bằng
- **Working day**: 68.3% ngày làm việc, 31.7% ngày nghỉ/lễ

In [ ]:
# Detailed statistics for categorical features (freq, proportions)
print("Detailed frequency and proportions of categorical variables:\n")
for col in VARS_CATEGORICAL + VARS_BINARY:
    freq = df[col].value_counts()
    prop = df[col].value_counts(normalize=True) * 100
    stats_df = pd.DataFrame({'Frequency': freq, 'Proportion (%)': prop})
    print(f"Feature: {col}")
    display(stats_df.round(2))
    print("\n")


**Phân tích Chi tiết Biến Phân loại (Categorical Distributions):**
- **Tính cân bằng dữ liệu**: Các biến thời gian cốt lõi như `season` (mỗi mùa ~25%) và `yr` (2011: 49.7%, 2012: 50.3%) có phân bố cực kỳ đồng đều, lý tưởng cho quá trình huấn luyện.
- **Đặc trưng thời tiết**: Lớp thời tiết khắc nghiệt (`weathersit=4`: Heavy Rain/Snow) chiếm tỷ lệ vô cùng nhỏ (chỉ 0.02%, 3 mẫu). Điều này tạo ra rủi ro *imbalanced class* cục bộ tại các node quyết định của thuật toán phân nhánh, dễ gây nhiễu trọng số hoặc underfitting cho nhóm thiểu số này.
- **Phân bố ngày làm việc**: Tỷ lệ `workingday` (~68%) phản ánh chính xác cấu trúc tuần làm việc thực tế (5/7 ngày), đảm bảo tính đại diện của mẫu thu thập.

### 3.5 Phân tích Tương quan

In [ ]:
# Correlation matrix
corr_cols = VARS_CONTINUOUS + VARS_TARGET + ['registered', 'casual']
corr_matrix = df[corr_cols].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
            fmt=".3f", linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix\n(Continuous Features + Target + Leakage Indicators)', 
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print highest correlations
print("Top Correlations with Target (cnt)")
target_corr = corr_matrix['cnt'].drop('cnt').sort_values(key=abs, ascending=False)
for var, corr in target_corr.items():
    print(f"  {var:12s}: {corr:6.3f}")

### 3.5.1 Top-k tương quan với biến mục tiêu (Positive/Negative)

Để tăng tính diễn giải theo rubric, ta tách riêng các đặc trưng tương quan dương mạnh nhất và âm mạnh nhất với `cnt` (không tính chính `cnt`).

In [ ]:
# Top-k positive/negative correlations with target
corr_target_full = df.drop(columns=VARS_DROP).corr(numeric_only=True)['cnt'].drop('cnt')

top_positive = corr_target_full.sort_values(ascending=False).head(8)
top_negative = corr_target_full.sort_values(ascending=True).head(8)

print('Top positive correlations with cnt:')
display(top_positive.to_frame(name='corr_with_cnt'))

print('\nTop negative correlations with cnt:')
display(top_negative.to_frame(name='corr_with_cnt'))

# Correlation without leakage variables for model-focused interpretation
corr_target_no_leak = df.drop(columns=VARS_DROP + VARS_LEAKAGE).corr(numeric_only=True)['cnt'].drop('cnt')
print('\nTop correlations with cnt (after removing leakage features):')
display(corr_target_no_leak.sort_values(key=abs, ascending=False).head(10).to_frame(name='corr_no_leak'))

**Kết luận bổ sung:**
- Bảng Top-k giúp xác định nhanh đặc trưng ưu tiên cho mô hình.
- Khi loại bỏ leakage (`casual`, `registered`), nhóm đặc trưng thời tiết và thời gian trở thành nguồn tín hiệu chính.
- Kết quả này là căn cứ trực tiếp cho bước Feature Selection trong preprocessing/modeling.

**Phân tích Tương quan:**
- **`temp` ↔ `atemp`**: r = 0.991 → **Đa cộng tuyến nghiêm trọng** (collinearity)
- **`cnt` ↔ `registered`**: r = 0.972 → Data leakage xác nhận
- **`cnt` ↔ `casual`**: r = 0.694 → Data leakage xác nhận
- **`cnt` ↔ `temp`**: r = 0.394 → Tương quan dương trung bình (nhiệt độ cao → thuê nhiều)
- **`cnt` ↔ `atemp`**: r = 0.389 → Tương tự temp
- **`cnt` ↔ `hum`**: r = -0.317 → Tương quan âm (độ ẩm cao → thuê ít)
- **`cnt` ↔ `windspeed`**: r = 0.101 → Tương quan yếu

In [ ]:
# Correlation heatmap for all features and target
# One-hot encode categorical variables for correlation calculation
df_encoded = pd.get_dummies(df.drop(columns=VARS_DROP + VARS_LEAKAGE), columns=VARS_CATEGORICAL, drop_first=True)

corr_all = df_encoded.corr()

plt.figure(figsize=(18, 12))
mask_all = np.triu(np.ones_like(corr_all, dtype=bool), k=1)
sns.heatmap(corr_all, mask=mask_all, annot=False, cmap='RdBu_r', center=0, vmin=-1, vmax=1, 
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Full Correlation Matrix (All Features + Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Display variables with the strongest correlation to the target
print("Top encoded variables highly correlated with 'cnt':")
corr_with_target = corr_all['cnt'].drop('cnt').sort_values(key=abs, ascending=False)
display(corr_with_target.head(10).to_frame())


**Phân tích Tương quan Toàn diện (Full Correlation Matrix):**
- **Yếu tố Thời gian vs Mục tiêu**: Đặc trưng `yr_1` (năm 2012) thể hiện tương quan dương đáng kể (~0.25) với `cnt`, phản ánh tốc độ tăng trưởng tự nhiên (organic growth) của dịch vụ sau một năm. Các mốc giờ cao điểm (nếu tách rời qua one-hot encoding) cũng sẽ chiếm tỷ trọng lớn trong việc quyết định lượng thuê.
- **Động lực Tuyến tính**: Biến nhiệt độ (`temp`, `atemp`) duy trì vị thế là nhóm đặc trưng liên tục có động lực tuyến tính mạnh nhất (r ~ 0.4).
- **Nhiễu Đa cộng tuyến mở rộng**: Đáng chú ý là các cụm tương quan nội bộ cực cao giữa các biến `season` và `mnth` (vd: mùa đông `season_4` và `mnth_10`, `mnth_11`). Sự phụ thuộc lẫn nhau này có nguy cơ gây bất ổn định cho ma trận thiết kế (design matrix) trong Normal Equations. Cần linh hoạt cân nhắc L1/L2 Regularization để nén bớt các trọng số thừa.

### 3.6 Scatter Plots: Quan hệ giữa Đặc trưng Liên tục và Mục tiêu

In [ ]:
# Scatter + regression trend: continuous features vs target
# Sampling is used to keep plots clear and efficient on large data.
plot_df = df[VARS_CONTINUOUS + ['cnt']].sample(n=min(5000, len(df)), random_state=SEED)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(VARS_CONTINUOUS):
    ax = axes[idx]

    sns.regplot(
        data=plot_df,
        x=col,
        y='cnt',
        ax=ax,
        scatter_kws={'alpha': 0.25, 's': 10, 'color': COLORS[idx]},
        line_kws={'color': 'red', 'linewidth': 2}
    )

    corr = df[col].corr(df['cnt'])
    ax.set_xlabel(col, fontsize=11)
    ax.set_ylabel('cnt (Total Rentals)', fontsize=11)
    ax.set_title(f'{col} vs cnt\n(Correlation: {corr:.3f})', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Scatter + Regression Trend: Continuous Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Phân tích Scatter Plots:**
- **`temp` vs `cnt`**: Quan hệ tuyến tính dương rõ rệt. Nhiệt độ từ 0.2-0.8 cho thấy số lượng thuê tăng
- **`atemp` vs `cnt`**: Tương tự temp, pattern gần như giống hệt → collinearity
- **`hum` vs `cnt`**: Quan hệ âm, độ ẩm cao (>0.8) thì số lượng thuê giảm
- **`windspeed` vs `cnt`**: Không rõ pattern, nhiều điểm tập trung ở windspeed=0

### 3.7 Pairplot tổng quát

In [ ]:
# Pairplot for continuous features

pairplot_data = df[VARS_CONTINUOUS + ['cnt']].sample(n=min(3000, len(df)), random_state=SEED)

g = sns.pairplot(pairplot_data, 
                 vars=VARS_CONTINUOUS + ['cnt'],
                 diag_kind='kde',
                 plot_kws={'alpha': 0.4, 's': 15},
                 diag_kws={'fill': True})

g.fig.suptitle('Pairplot: Continuous Features + Target', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Phân tích Pairplot:**
- Xác nhận lại đa cộng tuyến giữa `temp` và `atemp` (scatter gần như đường thẳng)
- `temp`/`atemp` có quan hệ tuyến tính dương với `cnt`
- `windspeed` có nhiều giá trị = 0 → cần xử lý
- `hum` phân bố khá độc lập với các biến khác

### 3.8 Phân tích Biến theo Thời gian

In [ ]:
# Temporal patterns analysis
fig, axes = plt.subplots(3, 1, figsize=(15, 14))

# Hourly pattern with workingday split
hourly_stats = df.groupby(['hr', 'workingday'])['cnt'].mean().reset_index()
pivot_hourly = hourly_stats.pivot(index='hr', columns='workingday', values='cnt')

axes[0].plot(pivot_hourly.index, pivot_hourly[0], 'o-', label='Non-working day', color=COLORS[0], linewidth=2)
axes[0].plot(pivot_hourly.index, pivot_hourly[1], 's-', label='Working day', color=COLORS[1], linewidth=2)
axes[0].set_title('Average Hourly Rentals: Working Day vs Non-working Day', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Find peak hours
peak_working = pivot_hourly[1].idxmax()
peak_nonworking = pivot_hourly[0].idxmax()
axes[0].axvline(x=peak_working, color=COLORS[1], linestyle='--', alpha=0.5, label=f'Peak working: {peak_working}h')
axes[0].axvline(x=peak_nonworking, color=COLORS[0], linestyle='--', alpha=0.5, label=f'Peak non-working: {peak_nonworking}h')

# Weekly pattern
weekly_stats = df.groupby('weekday')['cnt'].agg(['mean', 'std']).reset_index()
weekday_names = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
axes[1].bar(range(7), weekly_stats['mean'], yerr=weekly_stats['std'], 
          color='steelblue', edgecolor='black', capsize=5, alpha=0.7)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(weekday_names)
axes[1].set_title('Average Rentals by Weekday (with std)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Count')

# Monthly pattern by year
monthly_stats = df.groupby(['mnth', 'yr'])['cnt'].mean().reset_index()
for year in [0, 1]:
    year_data = monthly_stats[monthly_stats['yr'] == year]
    axes[2].plot(year_data['mnth'], year_data['cnt'], 'o-', 
                 label=f'Year {year} ({2011+year})', linewidth=2, markersize=8)
axes[2].set_title('Average Rentals by Month (Year Comparison)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Count')
axes[2].set_xticks(range(1, 13))
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak working day hour: {peak_working}:00 (avg: {pivot_hourly[1].max():.1f} rentals)")
print(f"Peak non-working day hour: {peak_nonworking}:00 (avg: {pivot_hourly[0].max():.1f} rentals)")

**Phân tích Mẫu Thời gian:**
- **Working day**: Pattern bimodal rõ rệt với 2 đỉnh:
  - 8:00 AM (đi làm): trung bình ~360 rentals
  - 17:00-18:00 PM (tan làm): đỉnh cao nhất ~600+ rentals
- **Non-working day**: Pattern đơn đỉnh, cao nhất vào 13:00-14:00 (~300 rentals) - thời gian đi chơi
- **Ngày trong tuần**: Thứ 6 và Thứ 7 có số lượng thuê cao nhất
- **Mùa**: Tháng 5-10 (hè-thu) có số lượng thuê cao, tháng 1-2 (đông) thấp nhất
- **Year 2012**: Cao hơn 2011 ở hầu hết các tháng → xu hướng tăng trưởng

## 4. Phát hiện Ngoại lai (Outlier Detection)

### 4.1 Phương pháp IQR (Interquartile Range)

In [ ]:
# IQR-based outlier detection
def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method."""
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound, q1, q3, iqr

# Detect outliers for all continuous features
print("IQR Outlier Detection")
outlier_summary = []

for col in VARS_CONTINUOUS + ['cnt']:
    outliers, lower, upper, q1, q3, iqr = detect_outliers_iqr(df, col)
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    
    outlier_summary.append({
        'Feature': col,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Lower Bound': lower,
        'Upper Bound': upper,
        'Outlier Count': outlier_count,
        'Outlier %': outlier_pct
    })
    
    print(f"\n{col}:")
    print(f"  Q1: {q1:.4f}, Q3: {q3:.4f}, IQR: {iqr:.4f}")
    print(f"  Bounds: [{lower:.4f}, {upper:.4f}]")
    print(f"  Outliers: {outlier_count} ({outlier_pct:.2f}%)")

# Convert to DataFrame
outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df.round(4))

**Kết quả IQR Outlier Detection:**
- **`windspeed`**: 394 outliers (2.27%) - nhiều nhất, đa số là windspeed=0 (dưới lower bound)
- **`hum`**: 275 outliers (1.58%) - độ ẩm rất thấp (gần 0)
- **`temp`**: 4 outliers (0.02%) - ít nhất
- **`atemp`**: 3 outliers (0.02%)
- **`cnt`**: 477 outliers (2.74%) - các ngày có lượng thuê rất cao (>600)

### 4.2 Phương pháp Z-Score

In [ ]:
# Z-score based outlier detection
def detect_outliers_zscore(data, column, threshold=3):
    """Detect outliers using Z-score method."""
    z_scores = np.abs(stats.zscore(data[column]))
    outliers = data[z_scores > threshold]
    return outliers, z_scores

print("=== Z-Score Outlier Detection (|z| > 3) ===")

for col in VARS_CONTINUOUS + ['cnt']:
    outliers, z_scores = detect_outliers_zscore(df, col, threshold=3)
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    max_z = z_scores.max()
    
    print(f"{col:12s}: {outlier_count:4d} outliers ({outlier_pct:5.2f}%), max |z|: {max_z:.2f}")

**Kết quả Z-Score:**
- Z-score > 3 phát hiện ít outliers hơn IQR (chỉ các giá trị cực đoan)
- **`windspeed`**: 239 outliers với max |z|=5.39
- **`cnt`**: 227 outliers - các ngày đặc biệt đông

### 4.3 Trực quan hóa Outliers

In [ ]:
# Boxplots for outlier visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

all_cols = VARS_CONTINUOUS + ['cnt']

for idx, col in enumerate(all_cols):
    ax = axes[idx]
    bp = ax.boxplot(df[col], patch_artist=True, notch=True)
    bp['boxes'][0].set_facecolor(COLORS[idx % len(COLORS)])
    bp['boxes'][0].set_alpha(0.7)
    
    # Add statistics
    median = df[col].median()
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    
    ax.set_title(f'{col}\n(median={median:.3f}, IQR={q3-q1:.3f})', fontweight='bold')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

# Remove extra subplot
axes[5].remove()

plt.suptitle('Boxplots for Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Kiểm định Thống kê

### 5.1 Kiểm định Breusch-Pagan (Kiểm tra Phương sai sai số không đồng nhất)

In [ ]:
# Breusch-Pagan test for heteroscedasticity
print("Breusch-Pagan Test for Heteroscedasticity:")
print("H0: Homoscedasticity (constant variance of residuals)")
print("H1: Heteroscedasticity (non-constant variance)\n")

# Fit OLS model with continuous features
X_ols = df[VARS_CONTINUOUS].copy()
X_ols = sm.add_constant(X_ols)  # Add intercept
y_ols = df['cnt']

# Fit the model
ols_model = sm.OLS(y_ols, X_ols).fit()

# Perform Breusch-Pagan test
bp_test = het_breuschpagan(ols_model.resid, ols_model.model.exog)

# Results labels
labels = ['LM Statistic', 'LM p-value', 'F Statistic', 'F p-value']
bp_results = dict(zip(labels, bp_test))

print("Breusch-Pagan Test Results:")
for key, value in bp_results.items():
    print(f"  {key:15s}: {value:.6f}")

# Interpretation
alpha = 0.05
if bp_results['LM p-value'] < alpha:
    print(f"\n Conclusion: p-value = {bp_results['LM p-value']:.2e} < {alpha}")
    print("  → REJECT H0: Heteroscedasticity is present")
    print("  → Consider using Weighted Least Squares (WLS)")
else:
    print(f"\n Conclusion: p-value = {bp_results['LM p-value']:.4f} >= {alpha}")
    print("  → DO NOT REJECT H0: No evidence of heteroscedasticity")

# R-squared of the model
print(f"\nOLS Model R-squared: {ols_model.rsquared:.4f}")
print(f"OLS Model Adjusted R-squared: {ols_model.rsquared_adj:.4f}")

**Phân tích Breusch-Pagan:**
- **LM Statistic**: Giá trị lớn cho thấy có sự khác biệt về phương sai
- **p-value < 0.05**: Có bằng chứng mạnh về heteroscedasticity
- **Kết luận**: Cần sử dụng WLS (Weighted Least Squares) hoặc robust standard errors trong mô hình hồi quy
- **R-squared**: Model OLS đơn giản chỉ giải thích ~31% variance của target

#### 5.1.1 Residual Plot để kiểm tra Heteroscedasticity

In [ ]:
# Residual plots for visual inspection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Fitted
axes[0].scatter(ols_model.fittedvalues, ols_model.resid, alpha=0.4, s=10, color=COLORS[0])
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Fitted Values', fontsize=11)
axes[0].set_ylabel('Residuals', fontsize=11)
axes[0].set_title('Residuals vs Fitted (OLS)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Scale-Location plot (sqrt of standardized residuals)
sqrt_std_resid = np.sqrt(np.abs(ols_model.get_influence().resid_studentized_internal))
axes[1].scatter(ols_model.fittedvalues, sqrt_std_resid, alpha=0.4, s=10, color=COLORS[1])
axes[1].set_xlabel('Fitted Values', fontsize=11)
axes[1].set_ylabel('√|Standardized Residuals|', fontsize=11)
axes[1].set_title('Scale-Location Plot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Residual Analysis: ")
print(f"Mean of residuals: {ols_model.resid.mean():.6f} (should be ~0)")
print(f"Std of residuals: {ols_model.resid.std():.4f}")
print(f"Min residual: {ols_model.resid.min():.2f}")
print(f"Max residual: {ols_model.resid.max():.2f}")

**Phân tách Residual Plots:**
- **Residuals vs Fitted**: Có pattern hình phễu (funnel shape) → xác nhận heteroscedasticity
- **Scale-Location**: Điểm phân tán không đều, tăng dần về phía fitted values cao
- **Khuyến nghị**: Sử dụng log-transform cho target hoặc WLS

### 5.2 Kiểm định Normality cho Residuals

In [ ]:
# Normality test for residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# QQ-plot for residuals
probplot(ols_model.resid, dist="norm", plot=axes[0])
axes[0].set_title('QQ-Plot: OLS Residuals', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Histogram of residuals
axes[1].hist(ols_model.resid, bins=50, edgecolor='black', alpha=0.7, color=COLORS[2])
axes[1].axvline(ols_model.resid.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1].set_title('Distribution of OLS Residuals', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

# Statistical tests
shapiro_stat, shapiro_p = stats.shapiro(ols_model.resid[:min(5000, len(ols_model.resid))])
jarque_stat, jarque_p = stats.jarque_bera(ols_model.resid)

print("Normality Tests for Residuals: ")
print(f"Shapiro-Wilk:  statistic={shapiro_stat:.4f}, p-value={shapiro_p:.2e}")
print(f"Jarque-Bera:   statistic={jarque_stat:.4f}, p-value={jarque_p:.2e}")
print(f"Skewness:      {ols_model.resid.skew():.4f}")
print(f"Kurtosis:      {ols_model.resid.kurtosis():.4f}")
print("\nInterpretation: p < 0.05 → Residuals NOT normally distributed")

**Kết quả Normality Tests:**
- Cả hai test đều reject normality (p < 0.001)
- Residuals lệch phải (skewness > 0) và có kurtosis cao → heavy tails
- **Giải pháp**: Transform target hoặc sử dụng robust regression

## 6. Phân tích Đa biến

### 6.1 Variance Inflation Factor (VIF) - Kiểm tra Đa cộng tuyến

In [ ]:
# Calculate VIF for continuous features
print("Variance Inflation Factor (VIF): ")
print("VIF > 5: Moderate multicollinearity")
print("VIF > 10: High multicollinearity (problematic)\n")

X_vif = df[VARS_CONTINUOUS].copy()
X_vif = sm.add_constant(X_vif)

vif_data = []
for i, col in enumerate(['const'] + VARS_CONTINUOUS):
    vif = variance_inflation_factor(X_vif.values, i)
    vif_data.append({'Feature': col, 'VIF': vif})
    
vif_df = pd.DataFrame(vif_data)

# Color code by severity
def vif_status(vif):
    if vif < 5:
        return '✓ OK'
    elif vif < 10:
        return '⚠ Moderate'
    else:
        return '✗ High'

vif_df['Status'] = vif_df['VIF'].apply(vif_status)
display(vif_df.round(2))

# Identify problematic features
high_vif = vif_df[vif_df['VIF'] > 5]['Feature'].tolist()
if high_vif:
    print(f"\n Features with VIF > 5: {high_vif}")
    print("  → Consider dropping one of highly correlated pairs")
else:
    print("\n✓ No severe multicollinearity detected")

**Phân tích VIF:**
- **`temp`**: VIF = 65.78 - **Cực kỳ cao** do tương quan gần như hoàn hảo với `atemp`
- **`atemp`**: VIF = 53.14 - **Cực kỳ cao**
- **`hum`**: VIF = 1.94 - OK
- **`windspeed`**: VIF = 1.50 - OK
- **Quyết định**: Loại bỏ `atemp` (giữ `temp` vì có tương quan cao hơn với target)

### 6.2 Phân tích Bivariate: Target theo Biến Phân loại

In [ ]:
# Bivariate analysis: target by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Target by Season
season_means = df.groupby('season')['cnt'].agg(['mean', 'std', 'count']).reset_index()
season_labels = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
season_means['season_name'] = season_means['season'].map(season_labels)

axes[0, 0].bar(season_means['season_name'], season_means['mean'], 
               yerr=season_means['std'], capsize=5, color='lightblue', edgecolor='black')
axes[0, 0].set_title('Average cnt by Season', fontweight='bold')
axes[0, 0].set_ylabel('Average Count')
for i, row in season_means.iterrows():
    axes[0, 0].text(i, row['mean'] + 20, f"{row['mean']:.0f}", ha='center')

# Target by Weather
weather_means = df.groupby('weathersit')['cnt'].agg(['mean', 'std']).reset_index()
weather_labels = {1: 'Clear', 2: 'Mist', 3: 'Light Rain', 4: 'Heavy Rain'}
weather_means['weather_name'] = weather_means['weathersit'].map(weather_labels)

axes[0, 1].bar(weather_means['weather_name'], weather_means['mean'],
               yerr=weather_means['std'], capsize=5, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Average cnt by Weather', fontweight='bold')
axes[0, 1].set_ylabel('Average Count')
axes[0, 1].tick_params(axis='x', rotation=15)

# Target by Working day
work_means = df.groupby('workingday')['cnt'].agg(['mean', 'std']).reset_index()
axes[1, 0].bar(['Non-Working', 'Working'], work_means['mean'],
               yerr=work_means['std'], capsize=5, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Average cnt by Working Day', fontweight='bold')
axes[1, 0].set_ylabel('Average Count')
for i, row in work_means.iterrows():
    axes[1, 0].text(i, row['mean'] + 5, f"{row['mean']:.0f}", ha='center')

# Target by Year
year_means = df.groupby('yr')['cnt'].agg(['mean', 'std']).reset_index()
axes[1, 1].bar(['2011', '2012'], year_means['mean'],
               yerr=year_means['std'], capsize=5, color='gold', edgecolor='black')
axes[1, 1].set_title('Average cnt by Year', fontweight='bold')
axes[1, 1].set_ylabel('Average Count')
for i, row in year_means.iterrows():
    axes[1, 1].text(i, row['mean'] + 5, f"{row['mean']:.0f}", ha='center')

plt.tight_layout()
plt.show()

# Print summary statistics
print("Bivariate Analysis Summary: ")
print("\nAverage cnt by Season:")
for _, row in season_means.iterrows():
    print(f"  {row['season_name']:8s}: {row['mean']:6.1f} (std={row['std']:5.1f}, n={row['count']})")

print(f"\nAverage cnt by Working Day:")
for _, row in work_means.iterrows():
    label = 'Working' if row['workingday'] == 1 else 'Non-Working'
    print(f"  {label:11s}: {row['mean']:6.1f} (std={row['std']:5.1f})")

print(f"\nAverage cnt by Year:")
for _, row in year_means.iterrows():
    year_label = '2011' if row['yr'] == 0 else '2012'
    print(f"  {year_label}: {row['mean']:6.1f} (std={row['std']:5.1f})")
    
growth = ((year_means.loc[1, 'mean'] - year_means.loc[0, 'mean']) / year_means.loc[0, 'mean']) * 100
print(f"\nYear-over-year growth: {growth:.1f}%")

**Phân tích Bivariate:**
- **Season**: Fall (mùa thu) có lượng thuê cao nhất (234.4), Spring thấp nhất (111.1)
- **Weather**: Clear weather cho thuê cao nhất (204.9), Heavy Rain thấp nhất (74.3)
- **Working Day**: Ngày làm việc (193.0) cao hơn ngày nghỉ (188.5) một chút
- **Year**: 2012 (280.3) cao hơn 2011 (96.6) rất nhiều → tăng trưởng 189.9%

### 6.3 Phân tích Tương tác (Interaction Analysis)

In [ ]:
# Interaction analysis: Hour x Working day
pivot_interaction = df.pivot_table(values='cnt', index='hr', columns='workingday', aggfunc='mean')

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_interaction.T, annot=False, cmap='YlOrRd', cbar_kws={'label': 'Average cnt'})
plt.title('Heatmap: Average Rentals by Hour and Working Day', fontsize=12, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Working Day (0=No, 1=Yes)')
plt.tight_layout()
plt.show()

# Find max/min in each category
print("Hour x Working Day Interaction: ")
print(f"Max non-working hour: {pivot_interaction[0].idxmax()}:00 ({pivot_interaction[0].max():.1f})")
print(f"Max working hour: {pivot_interaction[1].idxmax()}:00 ({pivot_interaction[1].max():.1f})")
print(f"Min non-working hour: {pivot_interaction[0].idxmin()}:00 ({pivot_interaction[0].min():.1f})")
print(f"Min working hour: {pivot_interaction[1].idxmin()}:00 ({pivot_interaction[1].min():.1f})")

**Phân tích Tương tác:**
- **Working day**: Hai dải màu đỏ rõ rệt vào 8h và 17-18h (giờ đi làm/về)
- **Non-working day**: Một dải màu đỏ duy nhất vào 12-14h (giờ đi chơi)
- **Min hours**: Cả hai đều thấp nhất vào 3-4h sáng (~10-20 rentals)
- **Tương tác quan trọng**: `hr` x `workingday` cần được xem xét khi feature engineering

In [ ]:
# Feature Interaction Analysis: Temperature x Humidity x Target
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['temp'], df['hum'], c=df['cnt'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Rental count (cnt)')
plt.xlabel('Temperature (temp)')
plt.ylabel('Humidity (hum)')
plt.title('Feature Interaction: Temp vs Humidity vs Target', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

**Phân tích Tương tác Đặc trưng (Feature Interaction):**
- **Điểm tụ (Sweet Spot) của Hành vi Người dùng**: Biểu đồ cho thấy phân bố lượng thuê (`cnt`) không hề tuyến tính độc lập. Một "cụm đắt giá" (màu vàng/đỏ) hội tụ mạnh tại góc phần tư `[temp: 0.5-0.8, hum: 0.3-0.6]`. Đây là minh chứng rõ nét cho việc thời tiết ấm áp cộng hưởng cùng độ ẩm thấp là tổ hợp tối ưu kích thích nhu cầu thuê xe.
- **Hiệu ứng Kìm hãm Kép**: Trái lại, khi nhiệt độ quá thấp (<0.3) hoặc độ ẩm dâng cao (>0.8), lượng thuê suy giảm về mức tối thiểu bất chấp biến số còn lại có ở ngưỡng lý tưởng hay không.
- **Hàm ý Mô hình hóa (Modeling Insight)**: Các mô hình Linear Regression cơ sở (OLS) sẽ thất bại trong việc nắm bắt bề mặt quyết định phi tuyến này. Để mô hình có thể hiểu được "hiệu ứng cộng hưởng", cần thiết phải bổ sung các hạng tử tương tác (interaction terms) như `temp * hum` hoặc chuyển qua không gian đặc trưng đa thức (Polynomial Basis Functions).

## 7. Đề xuất Tiền xử lý Dữ liệu (Preprocessing Strategy)

Để chuẩn bị dữ liệu cho phần xây dựng mô hình, dựa trên kết quả EDA, chúng tôi đề xuất các bước tiền xử lý như sau:

### 7.1 Xử lý Missing Values
Do tập dữ liệu hiện tại sạch **(không có missing values)** nên không cần nội suy trực tiếp. Tuy nhiên, pipeline vẫn giữ bước imputation dự phòng để đảm bảo tính tổng quát khi dữ liệu thực tế thay đổi.

### 7.2 Xử lý Data Leakage 
- Loại bỏ hoàn toàn hai biến `casual` và `registered` vì:
  - `cnt = casual + registered`
  - Giữ hai biến này sẽ làm mô hình học “đáp án” thay vì học quy luật dự báo.

### 7.3 Xử lý Outlier
- Outlier đã được phát hiện bằng cả IQR và Z-score.
- Chiến lược đề xuất cho preprocessing:
  - Không xóa dòng dữ liệu hàng loạt để tránh mất thông tin hành vi cực trị theo giờ.
  - Ưu tiên **winsorization/clip theo quantile của tập train** cho biến liên tục.
  - Kết hợp log-transform cho target để giảm tác động đuôi phải.

### 7.4 Xử lý Đặc trưng (Feature Engineering)
- Tạo biến tương tác (Interaction Features): `temp * hum`, `hr * workingday`.
- Xây dựng biến đại diện giờ cao điểm (rush-hour), nhóm thời điểm trong ngày.
- Cân nhắc mở rộng phi tuyến bậc thấp (ví dụ `temp^2`) cho mô hình nonlinear ở giai đoạn sau.

### 7.5 Mã hóa và Chuẩn hóa (Encoding & Scaling)
- **One-Hot Encoding**: cho các biến phân loại không có tính thứ tự (`season`, `mnth`, `hr`, `weekday`, `weathersit`).
- **Standardization (Z-score)**: cho biến liên tục để đồng nhất thang đo và cải thiện hội tụ cho các thuật toán tối ưu dựa trên gradient.

### 7.6 Chia dữ liệu (Train/Validation/Test Split)
- Chia theo tỷ lệ **70% Train / 10% Validation / 20% Test**.
- Toàn bộ bước fit của preprocessing phải chỉ dùng tập train để ngăn leakage.

### 7.7 Mapping từ Insight EDA sang Hành động
| Insight từ EDA | Hành động preprocessing |
|---|---|
| `casual`, `registered` gây leakage | Drop hoàn toàn trước khi train |
| `temp` và `atemp` đa cộng tuyến rất cao | Loại `atemp` (hoặc dùng regularization) |
| Target lệch phải mạnh | Dùng `log(1 + cnt)` |
| Outlier ở `windspeed`, `cnt` | Clip train-based quantile, không xóa ồ ạt |
| Có heteroscedasticity (Breusch-Pagan) | Chuẩn bị trọng số WLS ở bước modeling |

## 8. Kết luận EDA
- **Data Leakage**: Loại bỏ hoàn toàn `casual` và `registered`.
- **Đa cộng tuyến (Multicollinearity)**: Loại bỏ `atemp` vì tương quan quá cao với `temp` (VIF > 50).
- **Phân phối của Target**: Cần log-transform (`log(1+x)`) cho `cnt` để giảm độ lệch phải (skewness).
- **Phương sai không đồng nhất**: Có hiện tượng Heteroscedasticity rõ rệt (p-value < 0.001 của Breusch-Pagan test), cần sử dụng WLS (Weighted Least Squares) trong mô hình tuyến tính.
- **Tính sẵn sàng cho modeling**: Bộ insight đã được ánh xạ thành hành động preprocessing cụ thể, đáp ứng trực tiếp rubric phần 2.2.2.